In [10]:
import torch
import torch.nn as nn

mha = nn.MultiheadAttention(
    embed_dim=128,
    num_heads=8,
    batch_first=True,
    bias=False,
    device="cuda",
    dtype=torch.bfloat16,
)

In [11]:
import torch
import torch.nn.functional as F

class FlashMHA(torch.nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        self.qkv_proj = torch.nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = torch.nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.view(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # [3, B, H, T, D]
        q, k, v = qkv[0], qkv[1], qkv[2]  # each: [B, H, T, D]

        # FlashAttention!
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        out = out.transpose(1, 2).contiguous().view(B, T, C)  # [B, T, C]
        return self.out_proj(out)

In [14]:
model = FlashMHA(embed_dim=128, num_heads=8).to("cuda").to(torch.bfloat16)

data = torch.randn(10, 32, 128, device="cuda", dtype=torch.bfloat16)

with torch.autograd.profiler.profile(use_cuda=True) as prof:
    out = model(data)

print(prof.key_averages().table(sort_by="cuda_time_total"))

/tmp/ipykernel_1901261/2477043021.py:5: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:


---------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
---------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
           aten::scaled_dot_product_attention         4.75%      10.821ms        99.79%     227.319ms     227.319ms      10.824ms         4.75%     227.315ms     227.315ms             1  
    aten::_scaled_dot_product_flash_attention        32.53%      74.092ms        95.04%     216.494ms     216.494ms      74.115ms        32.54%     216.491ms     216.491ms             1  
               aten::_flash_attention_forward        26.51%

In [16]:
model = mha

data = torch.randn(10, 32, 128, device="cuda", dtype=torch.bfloat16)

with torch.autograd.profiler.profile(use_cuda=True) as prof:
    out = model(data, data, data)

print(prof.key_averages().table(sort_by="cuda_time_total"))

--------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                      Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
--------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
              aten::linear         2.51%      17.817us        36.22%     256.633us     128.317us      22.000us         3.19%     259.000us     129.500us             2  
              aten::matmul         3.02%      21.421us        26.97%     191.076us      95.538us      23.000us         3.33%     194.000us      97.000us             2  
               aten::clone         4.23%      29.989us        17.63%     124.933us      41.644us      32.000us         4.64%     129.000us      43.000us  

/tmp/ipykernel_1901261/4024781275.py:5: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

# Ensure FlashAttention is allowed
torch.backends.cuda.enable_flash_sdp(True)

d_model = 128
nhead = 8
dim_feedforward = 256
dropout = 0.0
num_layers = 2
device = "cuda"
dtype = torch.bfloat16

module = partial(
    nn.TransformerEncoderLayer,
    d_model=d_model,
    nhead=nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    bias=False,
    batch_first=True,
    norm_first=True,
    device=device,
    dtype=dtype,
    activation=F.gelu,
)

encoderlayers = nn.ModuleList([module() for _ in range(num_layers)])

# Dummy input
x = torch.randn(2, 32, d_model, device=device, dtype=dtype)

with torch.autograd.profiler.profile(use_cuda=True) as prof:
    out = encoderlayers[0](x)

print(prof.key_averages().table(sort_by="cuda_time_total"))

---------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
---------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                   aten::gelu         0.05%      33.665us        83.72%      59.690ms      59.690ms      59.697ms        83.74%      59.697ms      59.697ms             1  
                                    aten::add         0.07%      50.130us        12.93%       9.220ms       4.610ms       9.229ms        12.95%       9.229ms       4.614ms             2  
                             aten::layer_norm         0.03%

/tmp/ipykernel_1901261/2525414626.py:36: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:
